## algorithm design and anlysis-2025 spring  homework 4
**Deadline**：2025.5.14

**name**:


note：
---
1. 带有\*的题目，申请免上课的同学，必须完成，其他同学选作；
2. 请独立完成，如求助了他人或者大模型，请著明，并且不可省略算法分析部分；
4. 如若作答有雷同，全部取消成绩；
3. 需要书面作答的题目，可以通过引用图片的形式添加，但是注意上传项目时包含所引用的图片的源文件；
4. $log_n$ 默认表示$log_2{n}$;

## 问题 1 
**最小生成树（Minimum Spanning Tree）**

设  **G**  为一个带权重的连通无向图，且所有边的权重均不相等。令$e_i$ 为权重第 $i$ 小的边。最小生成树（MST）是否必须包含 $e_1$ ? 同理，是否必须包含 $e_2$ 和 $e_3$ ? 若必须包含，请给出证明；否则，请构造反例。需从基本原理论证，不能依赖割引理(cut lemma) 或 Prim/Kruskal算法的正确性。


answer:最小生成树是为了选出图中可以连接所有节点的边，并使得这些边的总权重之和最小，节点数为n的话，最小生成树的边数就是n-1。

首先，e1是权重最小的边。假设有两个节点A,B。原来e1是用来连接他们的，若选择其他权重边来代替e1，会造成总权重之和增加。根据最小生成树概念，应当要使权重和最小才是。换掉e1，便得不到最小权重和，所以e1无法被替代，必须包含。

然后，对于e2.假设有4个节点A,B,C,D。e1(A-B),e2(C-D),e3(B-C),e4(A-C),e5(B-D),可以发现，此时为了构建最小生成树，需要选择e1，e3，e5，此时权重和为9。第二种选择是选e2，e3，e4此时权重和也是9。可见，此时有两种方式构建最小生成树，所以e2不是必须包含的。

同理，对于e3.直接3个点A,B,C.e1(A-B),e2(B,C),e3(A-C),可见选e1，e2即可有权重和最小，所以e3也不是需要必须包含的。

## 问题 2 
**瓶颈生成树（Bottleneck Spanning Tree）**

带有权重的无向图 $G(V,E,w)$ 的瓶颈生成树，表现为：在所有生成树中，最大权重边的权重值最小。即，BST $T$ 最小化瓶颈损失 $c(T)=max_{e \in T}{w(e)}$。

1. 证明 $G$ 的每一个最小生成树（MST）都是瓶颈生成树（BST）
2. 设计一个线性时间复杂度的算法：， 对于一个图 $G(V,E,w)$ 和一个整数 $b$，判断图 $ G$ 是否存在一个瓶颈生成树，其最大权重边的权重不超过 $b$，分析算法设计思路，并基于python编程实现。
3. 设计一个线性时间复杂度的算法：对于给定的图 $G(V,E,w)$，找到其瓶颈生成树，分析算法设计思路，并基于python编程实现。

answer:1、T1为G的最小生成树。假设有生成树T2，其瓶颈损失比T1小，则T2的最大权重边比T1的小。通过使用T2中的一些边替换T1的一些权重大的边，就会形成T3。T3的最大权重边肯定比T1的小，然而T1是最小生成树，T3的权重和应该小于T1。这就造成矛盾，不可能存在T3权重和比T1小。这样写可以吗

idea：2、通过并查集判断图中是否存在瓶颈生成树。在过滤掉权重大于b的边后，使用并查集合并权重不超过b的边，看剩余边是否能够将所有节点连通。若连通，则就有瓶颈生成树。反之则没有。
有E条边，W是权重范围，则有复杂度O(ElogW)，当W范围较小，可以近似为O(E)。

此算法编写，参考自CSDN有关瓶颈生成树的文章，以及gpt的代码生成，在其基础上进行了一定的修改。

In [4]:
# add your code here
class DisjointSet:
    def __init__(self, num_nodes):
        # 初始化并查集
        self.parent = list(range(num_nodes))  # 每个节点的父节点指向自己
        self.rank = [0] * num_nodes  # 树的深度（用于优化合并操作）

    def find(self, node):
        # 查找节点的根节点，并使用路径压缩优化
        if self.parent[node] != node:
            self.parent[node] = self.find(self.parent[node])  # 路径压缩
        return self.parent[node]

    def union(self, node1, node2):
        # 合并两个集合
        root1 = self.find(node1)
        root2 = self.find(node2)

        if root1 != root2: 
            # 按秩合并：将深度小的树合并到深度大的树下面
            if self.rank[root1] > self.rank[root2]:
                self.parent[root2] = root1
            elif self.rank[root1] < self.rank[root2]:
                self.parent[root1] = root2
            else:
                self.parent[root2] = root1
                self.rank[root1] += 1

def can_form_bottleneck_spanning_tree(num_nodes, edge_list, max_weight):
    # num_nodes: 图中的节点数，edge_list: 边的列表
    dsu = DisjointSet(num_nodes) 
    
    # 遍历所有边，进行合并操作
    for node1, node2, weight in edge_list:
        if weight <= max_weight:  
            dsu.union(node1, node2)
    
    # 检查所有节点是否连通，即所有节点的根是否相同
    root_node = dsu.find(0)
    for i in range(1, num_nodes):
        if dsu.find(i) != root_node:
            return False  
    
    return True  

edges = [
    # 边 (u, v, w)
    (0, 1, 3),  
    (1, 2, 2),
    (2, 3, 4),
    (3, 4, 1),
    (0, 4, 5),
    (4, 5, 6),
    (5, 6, 7),
    (0, 2, 8),
    (1, 3, 9)
]
max_weight = 5  
num_nodes = 7  
print("判断图G是否存在一个瓶颈生成树，其最大权重边的权重不超过", max_weight)
print(can_form_bottleneck_spanning_tree(num_nodes, edges, max_weight))  
# algorithm of the liear time complexity 
'T(n)=O(E)'

判断图G是否存在一个瓶颈生成树，其最大权重边的权重不超过 5
False


'T(n)=O(E)'

idea：3、通过二分查找和并查集结合来寻找瓶颈生成树的最大边权。先用二分查找，在边的权重的范围内不断尝试，并在每次迭代中，检查图中所有边的权重小于等于当前中间值的边是否能够连接所有节点。然后用并查集判断图的连通性，最终确定瓶颈生成树的最大边权。
有E条边，W是权重范围，则有复杂度O(ElogW)，当W范围较小，可以近似为O(E)。

此算法编写，参考自CSDN有关瓶颈生成树的文章，以及gpt的代码生成，在其基础上进行了一定的修改。

In [7]:
# add your code here
class DisjointSet:
    def __init__(self, num_nodes):
        # 初始化并查集
        self.parent = list(range(num_nodes))  # 每个节点的父节点指向自己
        self.rank = [0] * num_nodes  # 树的深度（用于优化合并操作）

    def find(self, node):
        # 查找节点的根节点，并使用路径压缩优化
        if self.parent[node] != node:
            self.parent[node] = self.find(self.parent[node])  # 路径压缩
        return self.parent[node]

    def union(self, node1, node2):
        # 合并两个集合
        root1 = self.find(node1)
        root2 = self.find(node2)

        if root1 != root2: 
            # 按秩合并：将深度小的树合并到深度大的树下面
            if self.rank[root1] > self.rank[root2]:
                self.parent[root2] = root1
            elif self.rank[root1] < self.rank[root2]:
                self.parent[root1] = root2
            else:
                self.parent[root2] = root1
                self.rank[root1] += 1

def can_form_bottleneck_spanning_tree(num_nodes, edge_list, max_weight):
    # num_nodes: 图中的节点数，edge_list: 边的列表
    dsu = DisjointSet(num_nodes) 
    
    # 遍历所有边，进行合并操作
    for node1, node2, weight in edge_list:
        if weight <= max_weight:  
            dsu.union(node1, node2)
    
    # 检查所有节点是否连通，即所有节点的根是否相同
    root_node = dsu.find(0)
    for i in range(1, num_nodes):
        if dsu.find(i) != root_node:
            return False  
    return True  

def find_bottleneck_spanning_tree(num_nodes, edge_list):
    # 找到最大边权值的范围
    max_edge_weight = max(edge[2] for edge in edge_list)
    min_edge_weight = min(edge[2] for edge in edge_list)

    # 二分查找最大边权
    left, right = min_edge_weight, max_edge_weight
    answer = right

    while left <= right:
        mid = (left + right) // 2
        if can_form_bottleneck_spanning_tree(num_nodes, edge_list, mid):
            answer = mid  # 记录当前最大边权
            right = mid - 1  # 尝试更小的最大边权
        else:
            left = mid + 1  # 尝试更大的最大边权
    
    return answer

edges = [
    # 边 (u, v, w)
    (0, 1, 3),  
    (1, 2, 2),
    (2, 3, 4),
    (3, 4, 1),
    (0, 4, 5),
    (4, 5, 6),
    (5, 6, 7),
    (0, 2, 8),
    (1, 3, 9)
]
num_nodes = 7  # 图中的节点数

# 找到瓶颈生成树的最大边权
bottleneck_weight = find_bottleneck_spanning_tree(num_nodes, edges)
print("瓶颈生成树的最大权重边为: ",bottleneck_weight)
# algorithm of the liear time complexity 
'T(n)=O(E)'

瓶颈生成树的最大权重边为: 7


'T(n)=O(E)'

## 问题 3

**道路网（Road Network）**

假设有一个以图 $ G(V, E, l) $ 表示的道路网络，连接了一组城市 $ V $。我们假设该网络是有向的，并且每条道路 $(u, v) \in E$ 都有一个非负的长度 $ l(u, v) $。一条新的道路即将被建造，因此有一个列表 $ E' $ 包含它可以连接的城市对。每对 $(u, v) \in E'$ 都有一个对应的长度 $ l'(u, v) $。我们希望选择一对城市，使得两个城市 $ s, t \in V $ 之间的距离减少最大。请为此问题编写一个高效的算法，并详细解释算法的正确性和复杂度。


idea：使用Floyd-Warshall算法和路径更新评估。先用Floyd-Warshall算法，计算当前道路网络中所有城市对之间的最短路径。然后，对于对于每一对城市(start,end)，计算新道路带来的路径变化，判断是否能通过新道路缩短现有的最短路径。如果新路径更短，则记录下路径减少的量，并选择最大减少量的道路。
V是城市的数量，因为有三层循环，所以复杂度为O(V^3).

此算法编写，参考自CSDN有关Floyd-Warshall算法的文章，以及gpt的代码生成，在其基础上进行了一定的修改。

In [12]:
# add your code here
# 使用Floyd-Warshall计算所有城市对之间的最短路径
def calculate_shortest_paths(num_cities, road_network):
    shortest_paths = [[float('inf')] * num_cities for _ in range(num_cities)]
    
    # 初始化距离矩阵
    for start_city in range(num_cities):
        shortest_paths[start_city][start_city] = 0  # 自己到自己的距离为0
        for end_city, road_length in road_network[start_city]:
            shortest_paths[start_city][end_city] = road_length  # 图中的边赋予权重
    
    # 计算所有城市对之间的最短路径
    for k in range(num_cities):
        for i in range(num_cities):
            for j in range(num_cities):
                if shortest_paths[i][j] > shortest_paths[i][k] + shortest_paths[k][j]:
                    shortest_paths[i][j] = shortest_paths[i][k] + shortest_paths[k][j]
    
    return shortest_paths

# 计算通过新道路后，最短路径的变化
def evaluate_road_impact(num_cities, road_network, new_roads):
    shortest_paths = calculate_shortest_paths(num_cities, road_network)
    
    max_reduction = 0
    optimal_road = None
    
    # 遍历所有可选择的新道路
    for city_u, city_v, new_road_length in new_roads:
        # 对每一对城市对 (start, end)，计算新道路带来的路径变化
        for start in range(num_cities):
            for end in range(num_cities):
                # 计算通过新道路的路径
                new_path = shortest_paths[start][city_u] + new_road_length + shortest_paths[city_v][end]
                
                # 如果原路径不是 `inf`，并且新路径比原路径短，更新最大路径减少
                if shortest_paths[start][end] != float('inf') and new_path < shortest_paths[start][end]:
                    reduction = shortest_paths[start][end] - new_path
                    if reduction > max_reduction:
                        max_reduction = reduction
                        optimal_road = (city_u, city_v)
    
    return optimal_road, max_reduction

num_cities = 5
road_network = {
    0: [(1, 10), (2, 15)],  # 城市0连接到城市1和城市2，长度分别是10和15
    1: [(3, 5)],            # 城市1连接到城市3，长度为5
    2: [(3, 10)],           # 城市2连接到城市3，长度为10
    3: [(4, 2)],            # 城市3连接到城市4，长度为2
    4: []                   # 城市4没有连接任何其他城市
}
new_roads = [(0, 3, 8), (1, 4, 3)]  # 新道路的候选集

best_road, reduction_in_distance = evaluate_road_impact(num_cities, road_network, new_roads)
if best_road:
    print(f"最优选择的新道路是: {best_road}, 它减少的路径长度为: {reduction_in_distance}")
else:
    print("没有新道路能减少路径长度")
# your algorithm time complexity is:
'T(n)=O(V^3)'

最优选择的新道路是: (0, 3), 它减少的路径长度为: 7


'T(n)=O(V^3)'

## 问题 4

**逃离问题**

一个 $ n \times n $ 的网格是一个无向图，由 $ n $ 行和 $ n $ 列的顶点组成，如下图所示。我们用 $(i,j)$ 表示第 $ i $ 行和第 $ j $ 列的顶点。除了边界顶点，网格中的所有顶点都有四个邻居，即满足 $ i = 1, i = n, j = 1 $ 或 $ j = n $ 的点 $(i,j)$。

给定网格中的 $ m \leq n^2 $ 个起点 $(x_1, y_1), (x_2, y_2), \cdots , (x_m, y_m)$，逃离问题是确定是否存在 $ m $ 条顶点不相交的路径（即路径之间不相交），从这些起点到边界上的任意 $ m $ 个不同点。例如，图1中的网格存在逃离。

(1) 该问题可以看作是一个最大流问题。考虑一个流网络，其中顶点和边都有容量。也就是说，进入任何给定顶点的总正流量受到容量限制。证明在具有边和顶点容量的网络中确定最大流可以简化为在具有可比大小的普通流网络上的最大流问题。更准确地说，你需要将一个具有顶点和边容量的网络 $ G = (V,E) $ 转换为另一个仅具有边容量的网络 $ G' = (V', E') $，使得两个网络上的最大流相同，并且你构建的新网络具有 $ V' = O(V) $ 个顶点和 $ E' = O(E) $ 条边。你可以假设网络是连通的。

(2) 描述一个解决逃离问题的高效算法，并分析其运行时间。


<div align="center"> <img alt="图片" src="./fig/escepe-p.png"> </div>
<center> 图2. 逃脱问题网格，起始顶点为黑色，其他网格顶点为白色</center>

(1)的证明：先对每个顶点进行分割，每个顶点v分成两个部分：vin和vout，然后在这两个部分之间加一条边，边的容量就是原来顶点的容量。对于原网络中的每条边(u,v)，在新网络中创建一条从uout到vin的边，边的容量保持不变。这样，顶点容量就被转化为边容量限制，最终得到一个只有边容量的网络，且这个新网络的最大流和原始网络的最大流是一样的。新的网络顶点数和边数分别为O(V)和O(E)。

idea：(2)、通过将网格问题转化为最大流问题来解决逃离问题。先将网格中的每个点拆分为两个节点：输入端和输出端，并在它们之间添加容量为1的边，以确保每个点最多只能有一条路径通过。然后，连接相邻节点的输出端和输入端，边的容量设置为1。源点与起点之间、目标点与汇点之间分别建立容量为1的边。最终，使用Edmonds-Karp算法计算最大流，判断是否能找到不相交的路径。若最大流等于起点的数量，说明存在不相交路径。
时间复杂度主要由 Edmonds-Karp 最大流算法决定，该算法的时间复杂度为O(V⋅E^2)，V是顶点数，E是边数，对于n×n的网格，V=O(n^2)，E=O(n^2),所以整体应该是O(n^6)。

此算法编写，参考自CSDN有关Edmonds-Karp算法的文章，以及gpt的代码生成，在其基础上进行了一定的修改。

In [1]:
from collections import deque

# 使用 BFS 查找增广路径
def bfs(capacity, source, sink, parent):
    visited = [False] * len(capacity)
    queue = deque([source])
    visited[source] = True

    while queue:
        u = queue.popleft()

        for v in range(len(capacity)):
            if not visited[v] and capacity[u][v] > 0:  # 如果有剩余容量且未访问过
                queue.append(v)
                visited[v] = True
                parent[v] = u
                if v == sink:
                    return True
    return False

# 使用 Edmonds-Karp 算法计算最大流
def edmonds_karp(capacity, source, sink):
    parent = [-1] * len(capacity)
    max_flow = 0

    # 找到增广路径
    while bfs(capacity, source, sink, parent):
        path_flow = float('Inf')
        s = sink

        # 计算增广路径的最小容量
        while s != source:
            path_flow = min(path_flow, capacity[parent[s]][s])
            s = parent[s]

        # 更新剩余容量
        max_flow += path_flow
        v = sink
        while v != source:
            u = parent[v]
            capacity[u][v] -= path_flow
            capacity[v][u] += path_flow
            v = parent[v]

    return max_flow

# 构建网格图的流网络
def build_grid_network(n, start_points, boundary_points):
    # 节点数量是 n * n，每个节点分为 (i, j)_in 和 (i, j)_out，容量为 1
    total_nodes = 2 * n * n + 2  # 加上源点和汇点
    source = total_nodes - 2
    sink = total_nodes - 1

    # 创建容量矩阵
    capacity = [[0] * total_nodes for _ in range(total_nodes)]

    # 构建每个网格点的 (i, j)_in 和 (i, j)_out 之间的边
    for i in range(n):
        for j in range(n):
            in_node = i * n + j
            out_node = in_node + n * n
            capacity[in_node][out_node] = 1  # (i, j)_in 到 (i, j)_out 容量为 1

    # 为邻接节点之间添加边
    directions = [(-1, 0), (1, 0), (0, -1), (0, 1)]
    for i in range(n):
        for j in range(n):
            current_node_out = i * n + j + n * n
            for di, dj in directions:
                ni, nj = i + di, j + dj
                if 0 <= ni < n and 0 <= nj < n:
                    neighbor_node_in = ni * n + nj
                    capacity[current_node_out][neighbor_node_in] = 1  # 相邻点之间的容量为 1

    # 连接源点和起点
    for (x, y) in start_points:
        start_node_in = x * n + y
        capacity[source][start_node_in] = 1  # 源点到起点之间的容量为 1

    # 连接目标点和汇点
    for (x, y) in boundary_points:
        target_node_out = x * n + y + n * n
        capacity[target_node_out][sink] = 1  # 目标点到汇点之间的容量为 1

    return capacity, source, sink

# 逃离问题的解决函数
def solve_escape_problem(n, start_points, boundary_points):
    # 构建流网络
    capacity, source, sink = build_grid_network(n, start_points, boundary_points)
    
    # 使用 Edmonds-Karp 算法计算最大流
    max_flow = edmonds_karp(capacity, source, sink)
    
    # 如果最大流等于起点的数量，则表示可以找到不相交的路径
    if max_flow == len(start_points):
        return True  # 存在不相交的路径
    else:
        return False  # 不存在不相交的路径

# 示例网格，n=5
n = 5
start_points = [(0, 0), (1, 1), (2, 2)]  # 起点集合
boundary_points = [(0, 4), (4, 4), (4, 0)]  # 边界目标点集合

# 调用函数求解
result = solve_escape_problem(n, start_points, boundary_points)
print("是否存在不相交的路径:", result)
# your algorithm time complexity is:
'T(n)=O(n^6)'

是否存在不相交的路径: True


'T(n)=O(n^6)'